In [ ]:
gold_run_id = 'PASTE_GOLD_RUN_GUID'
ml_run_id = 'PASTE_NEW_ML_RUN_GUID'
coverage_start = 'YYYY-MM-DD'
complete_end = 'YYYY-MM-DD'
validation_start = 'YYYY-MM-DD'
test_start = 'YYYY-MM-DD'
model_name = 'gold_sales_qty_7d'
max_collect_rows = 2000000


In [ ]:
import json, uuid, hashlib
import datetime as dt
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
import notebookutils
from sklearn.ensemble import HistGradientBoostingRegressor
from mlflow.models import infer_signature
from pyspark.sql import functions as F
FEATURES = ['qty_lag_1', 'qty_sum_7d', 'qty_sum_28d', 'qty_avg_28d', 'qty_std_28d',
    'zero_days_28d', 'weekday', 'month']
FEATURE_VERSION = 'sales_qty_v1'

def require(ok, msg):
    if not bool(ok):
        raise ValueError(msg)

def observations(sales, elig, start, end, source_run_id):
    sales = sales.copy()
    elig = elig.copy()
    start = pd.Timestamp(start).normalize()
    end = pd.Timestamp(end).normalize()
    require(start <= end, 'Invalid observation coverage')
    require(not elig.product_key.duplicated().any(), 'Duplicate product eligibility')
    sales['business_date'] = pd.to_datetime(sales.business_date).dt.normalize()
    elig['active_start'] = pd.to_datetime(elig.active_start)
    elig['active_end'] = pd.to_datetime(elig.active_end)
    require(sales.source_gold_run_id.nunique() == 1 and str(sales.source_gold_run_id.iloc[0]) == source_run_id, 'Mixed or unexpected Gold versions')
    require(sales.order_qty.notna().all() and (sales.order_qty >= 0).all(),
        'Invalid sales quantity')
    elig = elig[elig.finished_goods_flag.astype(bool)]
    require(set(sales.product_key) <= set(elig.product_key),
        'Sales product outside eligible universe')
    groups = sales[(sales.business_date >= start) & (sales.business_date <= end)].groupby(['product_key', 'business_date']).agg(sold_qty=('order_qty', 'sum'), net_amount=('net_line_amount', 'sum')).reset_index()
    rows = []
    for r in elig.itertuples():
        require(pd.notna(r.active_start), 'Missing product start date')
        lo = max(start, r.active_start.normalize())
        hi = min(end, r.active_end.normalize()) if pd.notna(r.active_end) else end
        if lo <= hi:
            rows.append(pd.DataFrame({'product_key': int(r.product_key),
                'observation_date': pd.date_range(lo, hi)}))
    require(bool(rows), 'No eligible observations')
    grid = pd.concat(rows, ignore_index=True)
    require(set(map(tuple, groups[['product_key',
        'business_date']].to_numpy())) <= set(map(tuple, grid[['product_key',
        'observation_date']].to_numpy())), 'Sales outside eligibility dates')
    obs = grid.merge(groups, left_on=['product_key', 'observation_date'],
        right_on=['product_key', 'business_date'], how='left',
        validate='one_to_one').drop(columns='business_date')
    obs['sold_qty'] = obs.sold_qty.fillna(0).astype(float)
    obs['net_amount'] = obs.net_amount.fillna(0).astype(float)
    obs['availability_mode'] = 'RECONSTRUCTED'
    obs['source_gold_run_id'] = source_run_id
    return obs.sort_values(['product_key', 'observation_date']).reset_index(drop=True)

def features(obs):
    out = []
    for key, g in obs.groupby('product_key', sort=False):
        g = g.sort_values('observation_date').copy()
        require(not g.observation_date.duplicated().any(), 'Duplicate product-day')
        require(g.observation_date.diff().dropna().eq(pd.Timedelta(days=1)).all(),
            'Sparse calendar: ROW windows invalid')
        q = g.sold_qty.astype(float)
        prev = q.shift(1)
        g['qty_lag_1'] = prev
        g['qty_sum_7d'] = prev.rolling(7, min_periods=7).sum()
        g['qty_sum_28d'] = prev.rolling(28, min_periods=28).sum()
        g['qty_avg_28d'] = prev.rolling(28, min_periods=28).mean()
        g['qty_std_28d'] = prev.rolling(28, min_periods=28).std(ddof=0)
        g['zero_days_28d'] = prev.eq(0).astype(float).where(prev.notna()).rolling(28,
            min_periods=28).sum()
        g['weekday'] = g.observation_date.dt.dayofweek
        g['month'] = g.observation_date.dt.month
        g['label_qty_next_7d'] = q.iloc[::-1].rolling(7, min_periods=7).sum().iloc[::-1]
        g['label_end'] = g.observation_date + pd.Timedelta(days=6)
        g['baseline_qty'] = g.qty_avg_28d * 7
        out.append(g)
    return pd.concat(out, ignore_index=True)

def split_dataset(frame, val_start, test_start):
    f = frame.dropna(subset=FEATURES + ['label_qty_next_7d']).copy()
    v = pd.Timestamp(val_start)
    t = pd.Timestamp(test_start)
    require(v < t, 'Validation must precede test')
    train = f[f.label_end < v]
    val = f[(f.observation_date >= v) & (f.label_end < t)]
    test = f[f.observation_date >= t]
    for name, s in [('train', train), ('validation', val), ('test', test)]:
        require(len(s) > 0, 'Empty ' + name + ' after horizon purge')
    return (train, val, test)

def metrics(actual, pred):
    a = np.asarray(actual, float)
    p = np.asarray(pred, float)
    den = np.abs(a).sum()
    return {'mae': float(np.abs(a - p).mean()),
        'wape': float(np.abs(a - p).sum() / den) if den else None,
        'bias': float((p - a).mean()), 'n': int(len(a))}

def train(frame, val_start, test_start):
    tr, va, te = split_dataset(frame, val_start, test_start)
    model = HistGradientBoostingRegressor(loss='poisson', max_iter=100, max_depth=6,
        random_state=42)
    require(tr.label_qty_next_7d.sum() > 0, 'No positive training labels')
    model.fit(tr[FEATURES], tr.label_qty_next_7d)
    scores = {}
    for name, s in [('validation', va), ('test', te)]:
        pred = np.maximum(0, model.predict(s[FEATURES]))
        scores[name] = {'model': metrics(s.label_qty_next_7d, pred),
            'baseline': metrics(s.label_qty_next_7d, s.baseline_qty)}
    return (model, scores, dict(train=len(tr), validation=len(va), test=len(te)))

def dataset_id(obs):
    payload = obs.to_csv(index=False, float_format='%.10g',
        date_format='%Y-%m-%d').encode()
    return hashlib.sha256(payload).hexdigest()


In [ ]:
assert len(gold_run_id) == 36 and len(ml_run_id) == 36
assert 'YYYY' not in coverage_start + complete_end + validation_start + test_start
root = f'Files/gold_ml/{gold_run_id}'
ss = spark.read.parquet(f'{root}/sales')
es = spark.read.parquet(f'{root}/eligibility')
hs = spark.read.parquet(f'{root}/health')
assert ss.count() <= max_collect_rows, 'Sales export too large for driver'
assert es.count() <= 100000, 'Eligibility too large for tutorial'
assert hs.count() == 1, 'Expected one publication'
sales = ss.toPandas()
elig = es.toPandas()
health = hs.toPandas()
sales['order_qty'] = pd.to_numeric(sales['order_qty'])
sales['net_line_amount'] = pd.to_numeric(sales['net_line_amount'])
h = health.iloc[0]
assert str(h['run_id']) == gold_run_id, 'Mixed export versions'
assert pd.Timestamp(complete_end) <= pd.Timestamp(h['last_complete_date'])
out = f'Files/gold_ml_artifacts/{ml_run_id}'
assert not notebookutils.fs.exists(out), 'Use a new ML run ID'
notebookutils.fs.mkdirs(out)
print('Gold run', gold_run_id, 'sales rows', len(sales))


In [ ]:
toy = pd.DataFrame({'product_key': 1, 'observation_date': pd.date_range('2020-01-01',
    periods=80), 'sold_qty': np.arange(80, dtype=float), 'net_amount': 1.0,
    'availability_mode': 'RECONSTRUCTED', 'source_gold_run_id': gold_run_id})
a = features(toy)
assert a.loc[40, 'qty_sum_7d'] == toy.sold_qty.iloc[33:40].sum()
assert a.loc[40, 'label_qty_next_7d'] == toy.sold_qty.iloc[40:47].sum()
toy.loc[40:, 'sold_qty'] = 9999
b = features(toy)
assert a.loc[40, 'qty_sum_28d'] == b.loc[40, 'qty_sum_28d']
print('Temporal assertions passed')


In [ ]:
obs = observations(sales, elig, coverage_start, complete_end, gold_run_id)
assert len(obs) <= max_collect_rows, 'Observation grid too large'
frame = features(obs)
dsid = dataset_id(obs)
spark.createDataFrame(obs).write.mode('errorifexists').parquet(out + '/observations')
es.write.mode('errorifexists').parquet(out + '/eligibility')
x = frame[['product_key', 'observation_date'] + FEATURES]
y = frame[['product_key', 'observation_date', 'label_end', 'label_qty_next_7d']]
spark.createDataFrame(x).write.mode('errorifexists').parquet(out + '/features')
spark.createDataFrame(y).write.mode('errorifexists').parquet(out + '/labels')
print('dataset', dsid, 'observations', len(obs))


In [ ]:
model, scores, split_counts = train(frame, validation_start, test_start)
mlflow.set_experiment('exp_gold_sales_qty_7d')
sample = frame.dropna(subset=FEATURES).head(5)[FEATURES]
with mlflow.start_run() as active:
    mlflow.log_params({'dataset_id': dsid, 'source_gold_run_id': gold_run_id,
        'feature_version': FEATURE_VERSION, 'availability_mode': 'RECONSTRUCTED',
        'horizon_days': 7})
    mlflow.log_metrics({f'{s}_{kind}_{metric}': value for s,
        group in scores.items() for kind, m in group.items() for metric,
        value in m.items() if value is not None})
    mlflow.sklearn.log_model(model, 'model', signature=infer_signature(sample,
        model.predict(sample)))
    training_run_id = active.info.run_id
registered = mlflow.register_model(f'runs:/{training_run_id}/model', model_name)
training_result = {'ml_run_id': ml_run_id, 'gold_run_id': gold_run_id,
    'model_name': model_name, 'model_version': str(registered.version),
    'mlflow_run_id': training_run_id, 'dataset_id': dsid,
    'feature_version': FEATURE_VERSION, 'coverage_end': complete_end,
    'artifact_root': out, 'split_counts': split_counts, 'metrics': scores,
    'status': 'EXPERIMENTAL_NOT_APPROVED', 'availability_mode': 'RECONSTRUCTED'}
notebookutils.fs.put(out + '/training_result.json', json.dumps(training_result, indent=2),
    True)
print(json.dumps(training_result, indent=2))
